# LeetCode 2660 — Determine the Winner of a Bowling Game bằng Python

Link đề: https://leetcode.com/problems/determine-the-winner-of-a-bowling-game/ · Độ khó: **Easy** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def isWinner(self, player1: List[int], player2: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **mô phỏng (simulation)**: tính điểm của từng người chơi theo luật "lượt đánh sau khi đổ đủ `10` ky được nhân đôi trong hai lượt kế tiếp", rồi so sánh hai tổng điểm.
2. Xử lý **chỉ số lùi** `i - 1`, `i - 2` an toàn: biết vì sao `player[i - 1]` khi `i = 0` **không báo lỗi** mà lại lấy phần tử **cuối** (chỉ số âm trong Python) — một bẫy rất dễ trúng.
3. Dùng slicing `player[max(0, i - 2):i]`, toán tử `in`, biến đếm "còn bao nhiêu lượt được nhân đôi", và kỹ thuật **thêm phần tử đệm (padding)** ở đầu mảng.
4. Viết 5 cách: kiểm tra `i - 1`, `i - 2` có điều kiện biên, cắt lát `10 in player[max(0, i-2):i]`, bộ đếm lượt thưởng, đệm hai số `0` đầu mảng, và `zip` ba bản dịch của mảng; rồi đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: chỉ xét lượt `i - 1` mà quên `i - 2`, xét **lượt hiện tại** thay vì các lượt trước, bị wrap-around chỉ số âm, và trả sai mã kết quả (`1`, `2`, `0`).

## 2. Đề bài & ví dụ

Có hai mảng số nguyên `player1`, `player2` (chỉ số từ 0) ghi số ky (pin) mà người chơi 1 và người chơi 2 đổ được trong mỗi lượt. Trò chơi gồm `n` lượt; **mỗi lượt có đúng `10` ky**.

Giả sử một người đổ được `x_i` ky ở lượt `i`. **Giá trị của lượt `i`** với người đó là:

- `2 * x_i` nếu người đó **đổ đủ `10` ky ở lượt `i - 1` hoặc lượt `i - 2`**;
- ngược lại là `x_i`.

**Điểm** của một người là tổng giá trị `n` lượt của họ. Trả về:

- `1` nếu điểm của người chơi 1 **cao hơn**;
- `2` nếu điểm của người chơi 2 **cao hơn**;
- `0` nếu **hoà**.

| Ví dụ | player1, player2 | Output | Giải thích |
|---|---|---|---|
| 1 | `[5,10,3,2]`, `[6,5,7,3]` | `1` | P1: `5 + 10 + 2·3 + 2·2 = 25`; P2: `6 + 5 + 7 + 3 = 21`. |
| 2 | `[3,5,7,6]`, `[8,10,10,2]` | `2` | P1: `3 + 5 + 7 + 6 = 21`; P2: `8 + 10 + 2·10 + 2·2 = 42`. |
| 3 | `[2,3]`, `[4,1]` | `0` | `5` và `5` ⇒ hoà. |
| 4 | `[1,1,1,10,10,10,10]`, `[10,10,10,10,1,1,1]` | `2` | P1: `1+1+1+10+20+20+20 = 73`; P2: `10+20+20+20+2+2+1 = 75`. |

Constraints:

- `n == player1.length == player2.length`
- `1 <= n <= 1000`
- `0 <= player1[i], player2[i] <= 10`

Chú ý: nhân đôi chỉ phụ thuộc vào **hai lượt ngay trước** lượt đang tính, và vào việc lượt đó **đổ đủ 10 ky** — không phụ thuộc lượt hiện tại.

In [1]:
p1 = [5, 10, 3, 2]
print([f"lượt {i}: đổ {x}" for i, x in enumerate(p1)])
print("tổng nếu chưa nhân đôi:", sum(p1))

['lượt 0: đổ 5', 'lượt 1: đổ 10', 'lượt 2: đổ 3', 'lượt 3: đổ 2']
tổng nếu chưa nhân đôi: 20


## 3. Kiến thức nền

### 3.1. Chỉ số và chỉ số lùi

`player[i]` là số ky đổ ở lượt `i`; `player[i - 1]` là lượt **ngay trước**; `player[i - 2]` là lượt **trước đó nữa**. Với `i = 0` không có lượt nào trước; với `i = 1` chỉ có lượt `0` (không có lượt `-1`).

In [2]:
player = [5, 10, 3, 2]
i = 2
print("lượt", i, "=", player[i], "| lượt trước =", player[i - 1], "| trước nữa =", player[i - 2])

lượt 2 = 3 | lượt trước = 10 | trước nữa = 5


### 3.2. Bẫy: chỉ số âm không báo lỗi

Trong Python, `a[-1]` là **phần tử cuối**, `a[-2]` là phần tử áp cuối. Vậy khi `i = 0`, `player[i - 1]` là `player[-1]` — **không lỗi**, nhưng lấy sai phần tử (lượt cuối!). Ở bài này đó là bug âm thầm: lượt đầu có thể bị nhân đôi chỉ vì lượt **cuối** đổ đủ `10`.

In [3]:
player = [3, 4, 10]
print(player[0 - 1], player[0 - 2])   # 10 4 : lượt đầu "nhìn" nhầm vào lượt cuối!
print(player[-1], player[-3])         # 10 3

10 4
10 3


### 3.3. Cắt lát an toàn với `max(0, ...)`

Cắt lát (slicing) `a[l:r]` **không báo lỗi** khi `l` hay `r` vượt biên; nhưng nếu `l` âm thì lại tính từ cuối. Để lấy tối đa hai lượt trước lượt `i` mà không vượt đầu mảng: `player[max(0, i - 2):i]`.

- `i = 0`: `player[0:0]` = `[]`.
- `i = 1`: `player[0:1]` = lượt `0`.
- `i >= 2`: `player[i - 2:i]` = hai lượt trước.

In [4]:
player = [5, 10, 3, 2]
for i in range(4):
    print(f"i={i}: các lượt trước (tối đa 2) = {player[max(0, i - 2):i]}")
print(10 in [5, 10], 10 in [3, 2], 10 in [])   # True False False

i=0: các lượt trước (tối đa 2) = []
i=1: các lượt trước (tối đa 2) = [5]
i=2: các lượt trước (tối đa 2) = [5, 10]
i=3: các lượt trước (tối đa 2) = [10, 3]
True False False


### 3.4. Toán tử `in`, `or`, biểu thức điều kiện

- `10 in [5, 10]` kiểm tra `10` có thuộc danh sách (`True`).
- `A or B` đúng nếu A hoặc B đúng.
- `x * 2 if điều_kiện else x` là toán tử điều kiện: cho `2x` hoặc `x`.

In [5]:
x, doubled = 7, True
print(x * 2 if doubled else x)            # 14
doubled = False
print(x * 2 if doubled else x)            # 7

14
7


### 3.5. Phần tử đệm (padding) và `zip`

- **Đệm**: nối `[0, 0]` vào đầu mảng để mọi lượt thật đều có "hai lượt trước" (hai lượt giả `0` không phải là `10`). Khi đó ta không cần kiểm tra biên.
- `zip(a, b, c)` ghép các phần tử cùng vị trí; ghép `player`, `[0] + player`, `[0, 0] + player` cho từng lượt kèm lượt trước và trước nữa (dừng theo mảng ngắn nhất).

In [6]:
player = [5, 10, 3, 2]
print([0, 0] + player)                                   # [0, 0, 5, 10, 3, 2]
for x, prev1, prev2 in zip(player, [0] + player, [0, 0] + player):
    print(f"x={x:2}, lượt trước={prev1:2}, trước nữa={prev2:2}")

[0, 0, 5, 10, 3, 2]
x= 5, lượt trước= 0, trước nữa= 0
x=10, lượt trước= 5, trước nữa= 0
x= 3, lượt trước=10, trước nữa= 5
x= 2, lượt trước= 3, trước nữa=10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chia thành **hai việc**:

1. **Tính điểm một người** từ mảng số ky: với mỗi lượt `i`, nhìn hai lượt trước; nếu có lượt nào bằng `10` (strike) thì cộng `2·x_i`, ngược lại `x_i`.
2. **So sánh** hai điểm và trả `1`, `2` hoặc `0`.

Tính tay `player1 = [5, 10, 3, 2]`:

- Lượt 0: không có lượt trước ⇒ cộng `5`. (tổng `5`)
- Lượt 1: lượt 0 là `5` (không phải 10) ⇒ cộng `10`. (tổng `15`)
- Lượt 2: lượt 1 là `10` ⇒ **nhân đôi**: cộng `6`. (tổng `21`)
- Lượt 3: lượt 1 là `10` (cách hai lượt) ⇒ **nhân đôi**: cộng `4`. (tổng `25`)

Điểm `25`. Tương tự `player2 = [6, 5, 7, 3]` không có strike ⇒ `21`. Vì `25 > 21` ⇒ trả `1`.

In [7]:
def diem_tay(player):
    total = 0
    for i, x in enumerate(player):
        truoc = player[max(0, i - 2):i]
        doubled = 10 in truoc
        total += 2 * x if doubled else x
        print(f"lượt {i}: x={x}, hai lượt trước={truoc}, nhân đôi={doubled}, tổng={total}")
    return total

print(diem_tay([5, 10, 3, 2]))

lượt 0: x=5, hai lượt trước=[], nhân đôi=False, tổng=5
lượt 1: x=10, hai lượt trước=[5], nhân đôi=False, tổng=15
lượt 2: x=3, hai lượt trước=[5, 10], nhân đôi=True, tổng=21
lượt 3: x=2, hai lượt trước=[10, 3], nhân đôi=True, tổng=25
25


## 5. Cách 1 — Brute Force / baseline: kiểm tra `i - 1` và `i - 2` có điều kiện biên

**Ý tưởng:** viết hàm `score(player)`. Với mỗi `i`: đặt `doubled = False`; nếu `i >= 1` và `player[i - 1] == 10` thì `doubled = True`; nếu `i >= 2` và `player[i - 2] == 10` thì `doubled = True`. Cộng `2x` hay `x` tuỳ `doubled`. Rồi so sánh hai tổng.

Điều kiện `i >= 1`, `i >= 2` chính là **chống bẫy chỉ số âm** ở mục 3.2.

In [8]:
from typing import List

def score_checks(player: List[int]) -> int:
    total = 0
    for i in range(len(player)):
        doubled = False
        if i >= 1 and player[i - 1] == 10:
            doubled = True
        if i >= 2 and player[i - 2] == 10:
            doubled = True
        total += player[i] * 2 if doubled else player[i]
    return total

class SolutionChecks:
    def isWinner(self, player1: List[int], player2: List[int]) -> int:
        s1, s2 = score_checks(player1), score_checks(player2)
        if s1 > s2:
            return 1
        if s2 > s1:
            return 2
        return 0

In [9]:
sol = SolutionChecks()
print(sol.isWinner([5, 10, 3, 2], [6, 5, 7, 3]))                         # 1
print(sol.isWinner([3, 5, 7, 6], [8, 10, 10, 2]))                        # 2
print(sol.isWinner([2, 3], [4, 1]))                                      # 0
print(sol.isWinner([1, 1, 1, 10, 10, 10, 10], [10, 10, 10, 10, 1, 1, 1]))   # 2

1
2
0
2


**Độ phức tạp:** mỗi người một lượt duyệt `n` lượt, mỗi lượt `O(1)` ⇒ Time `O(n)`; Space `O(1)`.

**Nút thắt:** không có — đã tối ưu. Thuật toán chỉ có thể làm gọn hơn: cắt lát, bộ đếm, đệm, `zip`.

## 6. Ý tưởng thuật toán / bước đột phá

Hai cách nhìn khác để **bỏ điều kiện biên rườm rà**:

1. **Nhìn như "cửa sổ trượt"**: lượt `i` được nhân đôi khi `10` xuất hiện trong cửa sổ hai lượt trước `player[max(0, i-2):i]`. Một biểu thức `10 in ...` thay cho hai `if`.
2. **Nhìn như "bộ đếm thưởng"**: khi đổ đủ `10`, người chơi nhận **2 lượt thưởng** (lượt kế và lượt sau nữa). Mỗi lượt: nếu còn thưởng thì nhân đôi và giảm 1; nếu vừa đổ đủ `10` thì đặt lại thưởng `= 2`. Không cần nhìn lùi mảng — chỉ giữ **một số** `bonus`.
3. **Đệm đầu mảng**: thêm hai số `0` ở đầu để mọi lượt thật đều có hai lượt trước hợp lệ.

In [10]:
player = [10, 1, 1, 5]
bonus = 0
for x in player:
    doubled = bonus > 0
    print(f"x={x:2} bonus trước={bonus} -> {'nhân đôi' if doubled else 'giữ nguyên'}")
    bonus = 2 if x == 10 else max(0, bonus - 1)

x=10 bonus trước=0 -> giữ nguyên
x= 1 bonus trước=2 -> nhân đôi
x= 1 bonus trước=1 -> nhân đôi
x= 5 bonus trước=0 -> giữ nguyên


## 7. Các cách tiếp theo

### Cách 2 — Cắt lát `10 in player[max(0, i - 2):i]`

Gom hai điều kiện thành một biểu thức dùng `in` trên lát cắt. `max(0, i - 2)` ngăn chỉ số âm (nếu để `i - 2` âm, lát cắt `player[-1:i]` sẽ ra sai).

In [11]:
def score_slice(player: List[int]) -> int:
    return sum(x * 2 if 10 in player[max(0, i - 2):i] else x for i, x in enumerate(player))

class SolutionSlice:
    def isWinner(self, player1: List[int], player2: List[int]) -> int:
        s1, s2 = score_slice(player1), score_slice(player2)
        return 1 if s1 > s2 else 2 if s2 > s1 else 0

print(SolutionSlice().isWinner([5, 10, 3, 2], [6, 5, 7, 3]))     # 1
print(SolutionSlice().isWinner([3, 5, 7, 6], [8, 10, 10, 2]))    # 2
print(SolutionSlice().isWinner([2, 3], [4, 1]))                  # 0

1
2
0


**Độ phức tạp:** mỗi lát có tối đa 2 phần tử ⇒ Time `O(n)`, Space `O(1)` phụ (mỗi lát nhỏ, hằng số).

### Cách 3 — Bộ đếm lượt thưởng `bonus`

Giữ biến `bonus` (số lượt còn được nhân đôi). Mỗi lượt: nếu `bonus > 0` thì giá trị là `2x`; sau đó cập nhật: nếu `x == 10` thì `bonus = 2`, ngược lại `bonus = max(0, bonus - 1)`. **Chỉ một lượt duyệt, không nhìn lùi mảng.**

Kiểm tra: sau lượt đổ `10`, `bonus = 2` ⇒ lượt kế (`bonus 2 → 1`) và lượt sau nữa (`bonus 1 → 0`) đều nhân đôi; lượt thứ ba thì `bonus = 0` ⇒ không nhân đôi. Đúng "lượt `i - 1` hoặc `i - 2`".

In [12]:
def score_bonus(player: List[int]) -> int:
    total, bonus = 0, 0
    for x in player:
        total += 2 * x if bonus > 0 else x
        bonus = 2 if x == 10 else max(0, bonus - 1)
    return total

class SolutionBonus:
    def isWinner(self, player1: List[int], player2: List[int]) -> int:
        s1, s2 = score_bonus(player1), score_bonus(player2)
        return 1 if s1 > s2 else 2 if s2 > s1 else 0

print(SolutionBonus().isWinner([5, 10, 3, 2], [6, 5, 7, 3]))                         # 1
print(SolutionBonus().isWinner([1, 1, 1, 10, 10, 10, 10], [10, 10, 10, 10, 1, 1, 1]))   # 2

1
2


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

**Một chi tiết tinh tế:** nếu lượt kế tiếp lại là `10`, `bonus` được **đặt lại** về `2` (chứ không cộng dồn thành 3). Điều này đúng vì "nhân đôi" chỉ cần **một** lượt `i - 1` hoặc `i - 2` đủ `10`; nhân đôi nhiều lần cũng chỉ là `2x`, không phải `4x`.

### Cách 4 — Đệm hai số `0` ở đầu mảng

`p = [0, 0] + player`. Duyệt `i` từ `2` đến hết: lượt thật là `p[i]`, lượt trước `p[i - 1]`, trước nữa `p[i - 2]`. Vì hai số đệm là `0` (không phải `10`), không bao giờ gây nhân đôi nhầm; không cần kiểm tra biên.

In [13]:
def score_pad(player: List[int]) -> int:
    p = [0, 0] + player
    total = 0
    for i in range(2, len(p)):
        total += 2 * p[i] if p[i - 1] == 10 or p[i - 2] == 10 else p[i]
    return total

class SolutionPad:
    def isWinner(self, player1: List[int], player2: List[int]) -> int:
        s1, s2 = score_pad(player1), score_pad(player2)
        return 1 if s1 > s2 else 2 if s2 > s1 else 0

print(SolutionPad().isWinner([5, 10, 3, 2], [6, 5, 7, 3]))     # 1
print(SolutionPad().isWinner([3, 5, 7, 6], [8, 10, 10, 2]))    # 2

1
2


**Độ phức tạp:** Time `O(n)`; Space `O(n)` cho bản đệm (có thể làm `O(1)` bằng cách không copy, nhưng đơn giản hơn).

### Cách 5 — `zip` ba bản dịch của mảng

`zip(player, [0] + player, [0, 0] + player)` ghép sẵn `(x, lượt trước, trước nữa)`. Mỗi bộ cho một lượt; tổng được tính trong một biểu thức `sum`.

In [14]:
def score_zip(player: List[int]) -> int:
    return sum(
        x * 2 if prev1 == 10 or prev2 == 10 else x
        for x, prev1, prev2 in zip(player, [0] + player, [0, 0] + player)
    )

class SolutionZip:
    def isWinner(self, player1: List[int], player2: List[int]) -> int:
        s1, s2 = score_zip(player1), score_zip(player2)
        return (s1 > s2) + 2 * (s2 > s1)     # True/False cộng được: 1, 2 hoặc 0

print(SolutionZip().isWinner([5, 10, 3, 2], [6, 5, 7, 3]))     # 1
print(SolutionZip().isWinner([3, 5, 7, 6], [8, 10, 10, 2]))    # 2
print(SolutionZip().isWinner([2, 3], [4, 1]))                  # 0

1
2
0


**Độ phức tạp:** `zip` dừng ở mảng ngắn nhất (`player`, độ dài `n`) ⇒ Time `O(n)`; Space `O(n)` cho hai bản dịch.

Biểu thức cuối `(s1 > s2) + 2 * (s2 > s1)`: nếu `s1 > s2` là `1 + 0 = 1`; nếu `s2 > s1` là `0 + 2 = 2`; hoà là `0`.

## 8. TRACE từng bước

Trace Cách 3 (bộ đếm) cho `player2 = [10, 10, 10, 10, 1, 1, 1]` (ví dụ 4):

| lượt `i` | `x` | `bonus` trước | giá trị lượt | `bonus` sau | tổng |
|---|---|---|---|---|---|
| 0 | 10 | 0 | 10 | 2 (vừa đổ 10) | 10 |
| 1 | 10 | 2 | 20 | 2 | 30 |
| 2 | 10 | 2 | 20 | 2 | 50 |
| 3 | 10 | 2 | 20 | 2 | 70 |
| 4 | 1 | 2 | 2 | 1 | 72 |
| 5 | 1 | 1 | 2 | 0 | 74 |
| 6 | 1 | 0 | 1 | 0 | 75 |

Điểm `75`. Khớp lời giải của đề (`10 + 20 + 20 + 20 + 2 + 2 + 1`).

In [15]:
def trace_bonus(player):
    total, bonus = 0, 0
    for i, x in enumerate(player):
        doubled = bonus > 0
        value = 2 * x if doubled else x
        total += value
        new_bonus = 2 if x == 10 else max(0, bonus - 1)
        print(f"i={i} x={x:2} bonus={bonus} -> giá trị={value:2} | bonus mới={new_bonus} | tổng={total}")
        bonus = new_bonus
    return total

trace_bonus([10, 10, 10, 10, 1, 1, 1])

i=0 x=10 bonus=0 -> giá trị=10 | bonus mới=2 | tổng=10
i=1 x=10 bonus=2 -> giá trị=20 | bonus mới=2 | tổng=30
i=2 x=10 bonus=2 -> giá trị=20 | bonus mới=2 | tổng=50
i=3 x=10 bonus=2 -> giá trị=20 | bonus mới=2 | tổng=70
i=4 x= 1 bonus=2 -> giá trị= 2 | bonus mới=1 | tổng=72
i=5 x= 1 bonus=1 -> giá trị= 2 | bonus mới=0 | tổng=74
i=6 x= 1 bonus=0 -> giá trị= 1 | bonus mới=0 | tổng=75


75

In [16]:
def trace_checks(player):
    total = 0
    for i in range(len(player)):
        a = player[i - 1] if i >= 1 else None
        b = player[i - 2] if i >= 2 else None
        doubled = a == 10 or b == 10
        value = 2 * player[i] if doubled else player[i]
        total += value
        print(f"i={i} x={player[i]:2} lượt trước={a} trước nữa={b} nhân đôi={doubled!s:5} giá trị={value:2} tổng={total}")
    return total

trace_checks([1, 1, 1, 10, 10, 10, 10])

i=0 x= 1 lượt trước=None trước nữa=None nhân đôi=False giá trị= 1 tổng=1
i=1 x= 1 lượt trước=1 trước nữa=None nhân đôi=False giá trị= 1 tổng=2
i=2 x= 1 lượt trước=1 trước nữa=1 nhân đôi=False giá trị= 1 tổng=3
i=3 x=10 lượt trước=1 trước nữa=1 nhân đôi=False giá trị=10 tổng=13
i=4 x=10 lượt trước=10 trước nữa=1 nhân đôi=True  giá trị=20 tổng=33
i=5 x=10 lượt trước=10 trước nữa=10 nhân đôi=True  giá trị=20 tổng=53
i=6 x=10 lượt trước=10 trước nữa=10 nhân đôi=True  giá trị=20 tổng=73


73

## 9. Code nộp LeetCode

Dùng Cách 2: một hàm phụ tính điểm bằng cắt lát; kết quả so sánh rõ ràng.

In [17]:
from typing import List

class Solution:
    def isWinner(self, player1: List[int], player2: List[int]) -> int:
        def score(player: List[int]) -> int:
            total = 0
            for i, x in enumerate(player):
                if 10 in player[max(0, i - 2):i]:      # strike ở một trong hai lượt trước
                    total += 2 * x
                else:
                    total += x
            return total

        s1, s2 = score(player1), score(player2)
        if s1 > s2:
            return 1
        if s2 > s1:
            return 2
        return 0

In [18]:
print(Solution().isWinner([5, 10, 3, 2], [6, 5, 7, 3]))     # 1
print(Solution().isWinner([3, 5, 7, 6], [8, 10, 10, 2]))    # 2
print(Solution().isWinner([2, 3], [4, 1]))                  # 0
print(Solution().isWinner([1, 1, 1, 10, 10, 10, 10], [10, 10, 10, 10, 1, 1, 1]))   # 2

1
2
0
2


## 10. Vì sao thuật toán đúng

1. **Điểm một người.** Với lượt `i`, tập các lượt cần xét là `{i - 1, i - 2}` giao với `[0, n)`. Lát cắt `player[max(0, i - 2):i]` chứa đúng các chỉ số `max(0, i - 2), ..., i - 1` — chính là các lượt hợp lệ trong `{i - 2, i - 1}`. `10 in lát cắt` ⟺ có lượt trong hai lượt trước đổ đủ `10` ⟺ lượt `i` được nhân đôi, đúng luật.
2. **Không dùng lượt hiện tại để xét nhân đôi.** Lát cắt `[.. : i]` **không** gồm `player[i]`: một `strike` ở lượt `i` chỉ ảnh hưởng các lượt `i + 1`, `i + 2`, không tự nhân đôi chính nó.
3. **Chống chỉ số âm.** `max(0, i - 2)` đảm bảo cận trái không âm nên không "quay vòng" về cuối mảng.
4. **Tương đương của bộ đếm (Cách 3).** Quy nạp: ngay sau lượt đổ `10`, `bonus = 2`; lượt kế (cách `1`) nhận nhân đôi và `bonus` giảm xuống `1`; lượt sau nữa (cách `2`) vẫn nhân đôi và `bonus` về `0`; lượt cách `3` thì không. Nếu có `10` mới trong lúc đó, `bonus` được đặt lại `2` — luôn bằng `2 −` (số lượt kể từ strike gần nhất `− 1`) khi còn dương, nên `bonus > 0` ⟺ strike gần nhất cách `1` hoặc `2` lượt.
5. **So sánh.** Trả `1`, `2`, `0` đúng theo hơn / kém / bằng điểm.

## 11. Phân tích Time / Space Complexity

`n = len(player1) = len(player2)`.

| Cách | Time | Space |
|---|---|---|
| 1. Kiểm tra `i - 1`, `i - 2` | `O(n)` | `O(1)` |
| 2. Cắt lát `10 in ...` | `O(n)` | `O(1)` |
| 3. Bộ đếm `bonus` | `O(n)` | `O(1)` |
| 4. Đệm hai số `0` | `O(n)` | `O(n)` |
| 5. `zip` ba bản dịch | `O(n)` | `O(n)` |

Với `n <= 1000` mọi cách đều tức thì. Mọi cách đều tối ưu về bậc vì phải đọc mọi lượt.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: bốn ví dụ của đề; một lượt (`n = 1`) hoà / thắng / thua; strike ở lượt cuối (không ảnh hưởng ai); strike ở lượt đầu và `n = 2, 3`; hai strike liên tiếp; strike cách ba lượt (không nhân đôi); toàn `0`; toàn `10`.

In [19]:
sols = [SolutionChecks(), SolutionSlice(), SolutionBonus(), SolutionPad(), SolutionZip(), Solution()]
cases = [
    ("ví dụ 1", [5, 10, 3, 2], [6, 5, 7, 3], 1),
    ("ví dụ 2", [3, 5, 7, 6], [8, 10, 10, 2], 2),
    ("ví dụ 3", [2, 3], [4, 1], 0),
    ("ví dụ 4", [1, 1, 1, 10, 10, 10, 10], [10, 10, 10, 10, 1, 1, 1], 2),
    ("n=1 hoà", [10], [10], 0),
    ("n=1 P1 hơn", [5], [3], 1),
    ("n=1 P2 hơn", [0], [1], 2),
    ("strike ở lượt cuối", [1, 1, 10], [1, 1, 9], 1),
    ("strike lượt đầu, n=3", [10, 5, 5], [9, 5, 5], 1),
    ("strike cách ba lượt không nhân đôi", [10, 0, 0, 3], [10, 0, 0, 2], 1),
    ("toàn 0", [0, 0, 0], [0, 0, 0], 0),
    ("toàn 10", [10, 10, 10], [10, 10, 10], 0),
    ("P2 strike lượt đầu rồi toàn 0", [4, 4, 4], [10, 0, 0], 1),
    ("strike đầu, nhân đôi lượt 1", [4, 4, 4], [10, 5, 0], 2),
    ("strike đầu, nhân đôi lượt 2", [4, 4, 4], [10, 0, 5], 2),
]
for name, p1, p2, exp in cases:
    for s in sols:
        got = s.isWinner(p1[:], p2[:])
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 1
PASS ví dụ 2 -> 2
PASS ví dụ 3 -> 0
PASS ví dụ 4 -> 2
PASS n=1 hoà -> 0
PASS n=1 P1 hơn -> 1
PASS n=1 P2 hơn -> 2
PASS strike ở lượt cuối -> 1
PASS strike lượt đầu, n=3 -> 1
PASS strike cách ba lượt không nhân đôi -> 1
PASS toàn 0 -> 0
PASS toàn 10 -> 0
PASS P2 strike lượt đầu rồi toàn 0 -> 1
PASS strike đầu, nhân đôi lượt 1 -> 2
PASS strike đầu, nhân đôi lượt 2 -> 2


Kiểm tra tay vài ca:

- `[10, 5, 5]` vs `[9, 5, 5]`: P1 `10 + 2·5 + 2·5 = 30`; P2 `9 + 5 + 5 = 19` ⇒ `1`.
- `[4, 4, 4]` vs `[10, 0, 0]`: P1 `12`; P2 `10 + 0 + 0 = 10` ⇒ `1` (strike không giúp được nếu các lượt sau đều `0`).
- `[4, 4, 4]` vs `[10, 5, 0]`: P2 `10 + 2·5 + 0 = 20` ⇒ `2`. Còn `[10, 0, 5]`: P2 `10 + 0 + 2·5 = 20` (lượt `2` vẫn nằm trong hai lượt sau strike) ⇒ `2`.

In [20]:
print(Solution().isWinner([10, 5, 5], [9, 5, 5]))         # 1
print(Solution().isWinner([4, 4, 4], [10, 0, 0]))         # 1 (12 > 10)
print(Solution().isWinner([4, 4, 4], [10, 5, 0]))         # 2 (12 < 20)
print(Solution().isWinner([4, 4, 4], [10, 0, 5]))         # 2 (12 < 20)

1
1
2
2


### 12.2. Phản ví dụ cho lỗi chỉ số âm

Lỗi "wrap-around" ở mục 13 làm lượt đầu bị nhân đôi nhầm khi **lượt cuối** đổ `10`. Ta tạo ca để lời giải sai thất bại và chứng minh test của ta bắt được nó.

In [21]:
def sai_chi_so_am(player):
    total = 0
    for i, x in enumerate(player):
        doubled = player[i - 1] == 10 or player[i - 2] == 10   # i=0 -> player[-1], player[-2]
        total += 2 * x if doubled else x
    return total

p = [1, 1, 10]
print("sai :", sai_chi_so_am(p), "| đúng:", score_checks(p))
assert sai_chi_so_am(p) != score_checks(p)

sai : 14 | đúng: 12


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `n` ngẫu nhiên (`1..1000` — dùng nhỏ để oracle độc lập chạy nhanh và cả `n` lớn) và số ky `0..10` với xác suất `10` cao để hay có strike. Oracle viết bằng định nghĩa: với mỗi lượt kiểm tra tường minh các chỉ số `i - 1`, `i - 2` nếu hợp lệ.

In [22]:
import random
random.seed(2660)

def oracle_score(player):
    total = 0
    for i in range(len(player)):
        strike = False
        for j in (i - 1, i - 2):
            if 0 <= j < len(player) and player[j] == 10:
                strike = True
        total += player[i] * (2 if strike else 1)
    return total

def oracle(p1, p2):
    a, b = oracle_score(p1), oracle_score(p2)
    return 1 if a > b else (2 if b > a else 0)

def rand_player(n):
    return [10 if random.random() < 0.35 else random.randint(0, 10) for _ in range(n)]

for _ in range(600):
    n = random.randint(1, 12)
    p1, p2 = rand_player(n), rand_player(n)
    exp = oracle(p1, p2)
    for s in sols:
        got = s.isWinner(p1[:], p2[:])
        assert got == exp, (type(s).__name__, p1, p2, got, exp)
print("600 cặp ngẫu nhiên: mọi cách đều khớp oracle")

n = 1000
p1, p2 = rand_player(n), rand_player(n)
assert all(s.isWinner(p1[:], p2[:]) == oracle(p1, p2) for s in sols)
print("n = 1000 OK")

600 cặp ngẫu nhiên: mọi cách đều khớp oracle
n = 1000 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ xét lượt `i - 1`, quên lượt `i - 2`

In [23]:
from typing import List

def sai_thieu_i_tru_2(player: List[int]) -> int:
    total = 0
    for i, x in enumerate(player):
        doubled = i >= 1 and player[i - 1] == 10
        total += 2 * x if doubled else x
    return total

p = [5, 10, 3, 2]
print("sai :", sai_thieu_i_tru_2(p))           # 5 + 10 + 6 + 2 = 23
print("đúng:", score_checks(p))                # 25 (lượt 3 cũng nhân đôi vì strike ở lượt 1)

sai : 23
đúng: 25


**Vì sao sai:** đề nói "lượt `i - 1` **hoặc** lượt `i - 2`" — strike ảnh hưởng **hai** lượt sau. Ở ví dụ 1, lượt `3` (đổ `2`) phải tính `4`, không phải `2`.

### Lỗi 2: Nhân đôi lượt **hiện tại** khi nó là strike

In [24]:
def sai_luot_hien_tai(player: List[int]) -> int:
    total = 0
    for i, x in enumerate(player):
        total += 2 * x if x == 10 else x     # nhìn nhầm vào lượt hiện tại
    return total

p = [5, 10, 3, 2]
print("sai :", sai_luot_hien_tai(p))        # 5 + 20 + 3 + 2 = 30
print("đúng:", score_checks(p))             # 25

sai : 30
đúng: 25


**Vì sao sai:** strike ở lượt `i` làm **các lượt sau** (`i + 1`, `i + 2`) được nhân đôi, chứ **không** làm chính lượt `i` nhân đôi. Điều kiện luôn nhìn **về phía trước trong quá khứ** (các lượt trước).

### Lỗi 3: Bẫy chỉ số âm (wrap-around)

In [25]:
def sai_wrap(player: List[int]) -> int:
    total = 0
    for i, x in enumerate(player):
        doubled = player[i - 1] == 10 or player[i - 2] == 10     # không kiểm tra i >= 1 / i >= 2
        total += 2 * x if doubled else x
    return total

p = [1, 1, 10]
print("sai :", sai_wrap(p))        # lượt 0 và 1 bị nhân đôi nhầm vì player[-1] == 10
print("đúng:", score_checks(p))    # 1 + 1 + 10 = 12

sai : 14
đúng: 12


**Vì sao sai:** `player[-1]` là phần tử **cuối** nên với `i = 0` (và `i = 1` qua `player[-1]`) code âm thầm đọc lượt cuối. Không có lỗi nào được báo, chỉ có điểm sai. Luôn chặn biên: `i >= 1`, `i >= 2`, hoặc dùng `max(0, i - 2)`, hoặc đệm hai số `0`.

### Lỗi 4: Trả sai mã kết quả

In [26]:
def sai_ma_ket_qua(player1: List[int], player2: List[int]) -> int:
    s1, s2 = score_checks(player1), score_checks(player2)
    if s1 > s2:
        return 2            # đảo mã
    if s2 > s1:
        return 1
    return 0

print("sai :", sai_ma_ket_qua([5, 10, 3, 2], [6, 5, 7, 3]))     # 2
print("đúng:", Solution().isWinner([5, 10, 3, 2], [6, 5, 7, 3]))  # 1

sai : 2
đúng: 1


**Vì sao sai:** đề: `1` nếu **người chơi 1** có điểm cao hơn; `2` nếu người chơi 2 cao hơn; `0` hoà. Viết sẵn bảng trả về trước khi code để khỏi đảo nhầm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có "ảnh hưởng của vài phần tử **trước** đó" ⇒ **cửa sổ trượt** hoặc **bộ đếm "còn hiệu lực bao lâu"**.
- Mỗi khi viết `a[i - k]`, tự hỏi: `i < k` thì sao? Python cho chỉ số âm **không lỗi** nhưng sai nghĩa; dùng `max(0, i - k)` hoặc điều kiện `i >= k` hoặc đệm.
- Bài mô phỏng: nhập ví dụ, **tính tay** từng lượt theo luật rồi so với code; ví dụ 4 của đề rất tốt vì có nhiều strike liên tiếp.
- Kết quả "1/2/0 theo so sánh" ⇒ viết ba `if` hoặc `(a > b) + 2 * (b > a)`.
- Hai người chơi ⇒ viết **một** hàm `score` dùng hai lần, đừng sao chép code.

## 15. Biến thể của bài

1. **Cửa sổ hiệu lực `w`:** strike nhân đôi `w` lượt kế (thay vì `2`).
2. **Thưởng cộng dồn:** strike liên tiếp nhân ba, nhân tư ... (mô phỏng bowling thật).
3. **Trả về điểm từng lượt** thay vì chỉ người thắng.
4. **Game of Life / Simulation khác:** Baseball Game (LeetCode 682) — bảng điểm với quy tắc dựa trên các lượt trước.
5. **Best Time to Buy and Sell Stock, Running Sum (LeetCode 1480):** các bài "nhìn lại quá khứ" tương tự.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra `i - 1`, `i - 2` | Hai `if` có điều kiện biên | `O(n)` | `O(1)` | Dễ hiểu, rõ biên |
| 2. Cắt lát `10 in ...` | Cửa sổ hai lượt trước | `O(n)` | `O(1)` | **Nên nộp**: ngắn, rõ nghĩa |
| 3. Bộ đếm `bonus` | Giữ số lượt thưởng còn lại | `O(n)` | `O(1)` | Không nhìn lùi mảng; mở rộng dễ |
| 4. Đệm hai số `0` | Bỏ kiểm tra biên | `O(n)` | `O(n)` | Khi muốn code đồng nhất |
| 5. `zip` ba bản dịch | Ghép sẵn lượt trước | `O(n)` | `O(n)` | Phong cách Python hàm |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[5, 10, 3, 2][-1]`, `[1, 2, 3][max(0, -1):1]`, `10 in [3, 10][0:1]`, `(3 > 2) + 2 * (2 > 3)`.

**Bài 2 (tính tay).** `player1 = [10, 10, 2]`, `player2 = [9, 9, 9]`. Viết giá trị từng lượt của hai người, tổng điểm, và kết quả.

**Bài 3 (điền TODO).** Hoàn thành `score_todo(player)` dùng bộ đếm `bonus`.

In [27]:
def score_todo(player):
    total, bonus = 0, 0
    for x in player:
        # TODO 1: nếu bonus > 0 thì cộng 2 * x, ngược lại cộng x
        # TODO 2: cập nhật bonus: nếu x == 10 thì bonus = 2, ngược lại bonus = max(0, bonus - 1)
        pass
    return total

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `score_window(player, w)` trong đó một strike làm `w` lượt kế tiếp được nhân đôi (đề gốc ứng với `w = 2`). Kiểm tra `w = 2` khớp `score_checks`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `turn_values(player)` trả danh sách giá trị từng lượt. (b) Viết `winner_by_turns(p1, p2)` trả số lượt mà người chơi 1 **dẫn** tổng điểm tích luỹ (đếm các lượt `i` mà tổng đến `i` của P1 lớn hơn của P2).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[-1]` là phần tử cuối (`2`); `max(0, -1) = 0` nên lát `[0:1]` là `[1]`; `[3, 10][0:1] = [3]` không chứa `10`; `True + 2 * False = 1`. Bài 2: P1: lượt 0 `10`; lượt 1 (strike trước) `20`; lượt 2 (strike ở 0 và 1) `4`. P2: `9 + 9 + 9`.
- **Mức 2 (vừa):** Bài 3: `total += 2 * x if bonus > 0 else x`; `bonus = 2 if x == 10 else max(0, bonus - 1)`. Bài 4: tổng quát hoá `2` thành `w` ở chỗ đặt `bonus = w`.
- **Mức 3 (sâu):** Bài 5(a): giống `score` nhưng `append` từng giá trị vào danh sách. Bài 5(b): dùng `itertools.accumulate` hoặc hai biến tổng chạy song song rồi so sánh sau mỗi lượt.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert [5, 10, 3, 2][-1] == 2
assert [1, 2, 3][max(0, -1):1] == [1]
assert (10 in [3, 10][0:1]) is False
assert (3 > 2) + 2 * (2 > 3) == 1

# Bài 2
def values_checks(player):
    vals = []
    for i, x in enumerate(player):
        vals.append(2 * x if 10 in player[max(0, i - 2):i] else x)
    return vals

assert values_checks([10, 10, 2]) == [10, 20, 4]
assert values_checks([9, 9, 9]) == [9, 9, 9]
assert sum(values_checks([10, 10, 2])) == 34 and sum([9, 9, 9]) == 27
assert Solution().isWinner([10, 10, 2], [9, 9, 9]) == 1

# Bài 3
def score_todo_ans(player):
    total, bonus = 0, 0
    for x in player:
        total += 2 * x if bonus > 0 else x
        bonus = 2 if x == 10 else max(0, bonus - 1)
    return total

assert score_todo_ans([5, 10, 3, 2]) == 25
assert score_todo_ans([10, 10, 10, 10, 1, 1, 1]) == 75

# Bài 4
def score_window(player, w):
    total, bonus = 0, 0
    for x in player:
        total += 2 * x if bonus > 0 else x
        bonus = w if x == 10 else max(0, bonus - 1)
    return total

for p in ([5, 10, 3, 2], [1, 1, 1, 10, 10, 10, 10], [10, 0, 0, 0]):
    assert score_window(p, 2) == score_checks(p)
assert score_window([10, 1, 1, 1], 3) == 10 + 2 + 2 + 2
assert score_window([10, 1, 1, 1], 1) == 10 + 2 + 1 + 1

# Bài 5
def turn_values(player):
    vals = []
    bonus = 0
    for x in player:
        vals.append(2 * x if bonus > 0 else x)
        bonus = 2 if x == 10 else max(0, bonus - 1)
    return vals

assert turn_values([5, 10, 3, 2]) == [5, 10, 6, 4]

def winner_by_turns(p1, p2):
    a, b = 0, 0
    lead = 0
    for u, v in zip(turn_values(p1), turn_values(p2)):
        a += u
        b += v
        if a > b:
            lead += 1
    return lead

assert winner_by_turns([5, 10, 3, 2], [6, 5, 7, 3]) == 3     # tích luỹ: P1 5,15,21,25 vs P2 6,11,18,21
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hàm `score` dùng cửa sổ hai lượt trước)

```python
from typing import List

class Solution:
    def isWinner(self, player1: List[int], player2: List[int]) -> int:
        def score(player: List[int]) -> int:
            total = 0
            for i, x in enumerate(player):
                if 10 in player[max(0, i - 2):i]:      # strike ở một trong hai lượt trước
                    total += 2 * x
                else:
                    total += x
            return total

        s1, s2 = score(player1), score(player2)
        return 1 if s1 > s2 else 2 if s2 > s1 else 0
```

**Checklist**

- [ ] Nhân đôi nếu strike ở lượt `i - 1` **hoặc** `i - 2` (cả hai, không chỉ một)?
- [ ] Không xét lượt hiện tại để nhân đôi chính nó?
- [ ] Chặn chỉ số âm (`max(0, i - 2)`, `i >= 1`, hoặc đệm)?
- [ ] Mã kết quả đúng: `1` nếu P1 hơn, `2` nếu P2 hơn, `0` nếu hoà?
- [ ] Đã thử `n = 1`, strike ở lượt đầu và lượt cuối, hai strike liên tiếp, toàn `10`?
- [ ] Signature đúng `isWinner(self, player1: List[int], player2: List[int]) -> int` và có `from typing import List`?